### Maze-Generation
<img src='/files/images/maze_generation.png'>  

Wir betrachten folgendes Spiel auf einem (m x n)-Gitter.
Jedes Gitterfeld ist von 4 Wänden eingeschlossen.  
Der Spieler startet auf Feld (0, 0). Das Spiel enden, wenn er alle Felder besucht hat.
- Der Spieler kann in eine Wand eine Türe einbauen, um auf
  ein noch **unbesuchtes Nachbarfeld** zu gelangen.
- Der Spieler kann alle bestehenden Türe nutzen, um auf ein bereits besuchtes Feld zurückkehren. 



Wir benutzen eine Variante unseres Algorithmus zum Finden einer Zusammenhangskomponente, um
alle Felder zu besuchen. Wir speichern
alle **frisch besuchten** Felder in einer Liste `stack`, und alle
**besuchten Felder** in einer Menge `visited`.

- Zu Beginn ist `stack = [(0, 0)]` und `visited = {(0, 0)}`.
- Solange noch Felder zu besuchen sind, machen wir folgendes:
  - wir **entfernen** das letzte Feld der Liste Stack und gehen auf dieses Feld.
  - wir besuchen **von diesem Feld aus**, in zufälliger Reihenfolge, alle **unbesuchten** Nachbarfelder und
    fügen diese Felder zu `visited` und `stack` hinzu.

In [1]:
import random


def get_random_neighbors(pos, dims):
    directions = [(0, 1), (0, -1), (1, 0), (-1, 0)]
    random.shuffle(directions)

    x, y = pos
    for dx, dy in directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < dims[0] and 0 <= ny < dims[1]:
            yield (nx, ny)


def get_moves(dims, start=(0, 0)):
    stack = [(start)]
    visited = {start}

    while len(visited) < dims[0]*dims[1]:
        pos = stack.pop()
        for npos in get_random_neighbors(pos, dims):
            if npos in visited:
                continue
            visited.add(npos)
            stack.append(npos)
            yield pos, npos

In [2]:
list(get_random_neighbors((0, 0), (3, 3)))

[(0, 1), (1, 0)]

In [3]:
list(get_moves(dims=(4, 2)))

[((0, 0), (0, 1)),
 ((0, 0), (1, 0)),
 ((1, 0), (2, 0)),
 ((1, 0), (1, 1)),
 ((1, 1), (2, 1)),
 ((2, 1), (3, 1)),
 ((3, 1), (3, 0))]

In [4]:
import widget_helpers as W
import grid_helpers as G
from IPython.display import display


def init_canvas(ncol, nrow, dx=20, dy=20):
    canvas = W.get_canvas(width=ncol*dx, height=nrow*dy)
    grid_spec = (0, 0, dx, dy, ncol, nrow)
    G.draw_grid(canvas, grid_spec)

    canvas.stroke_style = 'blue'
    canvas.line_width = 3
    return canvas, grid_spec


def connect(canvas, src, dst, grid_spec):
    p = G.cr2xy(*src, grid_spec, center=True)
    q = G.cr2xy(*dst, grid_spec, center=True)
    canvas.stroke_line(*p, *q)


def draw_maze(ncol, nrow, dx=20, dy=20):
    canvas, grid_spec = init_canvas(ncol, nrow, dx, dy)
    for src, dst in get_moves(dims=(ncol, nrow)):
        connect(canvas, src, dst, grid_spec)
    display(canvas)

In [5]:
draw_maze(20, 10)

Canvas(height=200, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right=…